In [ ]:
import pandas as pd
import xgboost as xgb
import shap
from pyprojroot import here
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


# หาตำแหน่งของโฟลเดอร์ที่ไฟล์ code นี้วางอยู่
base_path = here()
file_path = os.path.join(base_path, "data", "SFProgramDataPanal.csv")
# file_path = os.path.join(base_path, "data", "SFProgramDataPanal.csv")

df = pd.read_csv(file_path)

# df.head()
print(df.head().to_markdown())

# 1. Load Data (สมมติว่าเป็นชุดข้อมูลเกษตรกรที่เราคลีนแล้ว)
# df = pd.read_csv('~/data/SFProgramDataPanal.csv')



In [ ]:
# 1. คัดเลือก Features ตามโครงสร้าง Column ของคุณ
# ผมแยกเป็นกลุ่มเพื่อให้โค้ดอ่านง่ายและแก้ไขตามทฤษฎีเศรษฐศาสตร์ได้สะดวก
human_capital = ['age', 'edu', 'agri_long']
production_assets = ['land_crop1_in', 'irriga', 'gov_support']
financial_risk = ['loan', 'Non-Farm Act', 'Overview_Risk']
social_capital = ['agri Org_mem']
baseline_skills = ['Avg_ProdManage', 'Avg_InputManage', 'Avg_Tech', 'Avg_Ana&Plan', 'Avg_Mkting', 'Avg_Network']

In [ ]:
# รวม Features ทั้งหมด
features = human_capital + production_assets + financial_risk + social_capital + baseline_skills

# 2. เตรียมข้อมูล (สมมติว่า df คือ DataFrame ที่โหลดมาจากไฟล์ของคุณ)
# ในขั้นตอนจริง ต้องมีการจัดการ Missing Value และแปลงค่า '.' เป็น NaN ก่อน
X = df[features].apply(pd.to_numeric, errors='coerce').fillna(0)

# 3. กำหนด Target (เป้าหมาย)
# ในกรณีนี้เราจะพยากรณ์ว่า "ทักษะจะเพิ่มขึ้นหรือไม่" (Ch_Skill > 0)
# หรือจะใช้ 31.agri_tech_adapt (มีการปรับใช้เทคโนโลยี) ก็ได้ครับ
y = (df['Ch_Skill'].apply(pd.to_numeric, errors='coerce') > 0).astype(int)

# 4. สร้างโมเดลพยากรณ์
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5)
model.fit(X_train, y_train)

# 5. การอธิบายผลด้วย SHAP (หัวใจสำคัญของเล่มวิจัย)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# สรุปผลว่าปัจจัยใด (เช่น ชลประทาน หรือ อายุ) ส่งผลต่อการเพิ่มทักษะมากที่สุด
print("โมเดลพร้อมสำหรับการวิเคราะห์ปัจจัยเชิงนโยบายแล้วครับ")

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from pyprojroot import here
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


# หาตำแหน่งของโฟลเดอร์ที่ไฟล์ code นี้วางอยู่
base_path = here()
file_path = os.path.join(base_path, "data", "SFProgramDataPanal.csv")

# โหลดข้อมูลจากไฟล์ CSV
df = pd.read_csv(file_path)

def clean_agricultural_data(df):
    # 1. จัดการสัญลักษณ์พิเศษ: เปลี่ยน '.' เป็น NaN เพื่อให้คำนวณทางสถิติได้
    df = df.replace('.', np.nan)
    
    # 2. คัดเลือก Features ตามโครงสร้าง Column ในงานวิจัย
    feature_cols = [
        'age', 'edu', 'agri_long', 'land_crop1_in', 'irriga', 
        'gov_support', 'loan', 'Non-Farm Act', 'other_inc', 'agri Org_mem',
        'Avg_ProdManage', 'Avg_InputManage', 'Avg_Tech', 'Avg_Ana&Plan', 'Avg_Mkting', 'Avg_Network'
    ]
    
    # 3. แปลงชนิดข้อมูลเป็น Numeric (ตัวเลข)
    for col in feature_cols + ['Ch_Skill']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # 4. จัดการค่าว่าง (Imputation)
    # สำหรับทุนและรายได้ เติม 0 (สมมติว่าถ้าว่างคือไม่มี) 
    # สำหรับทักษะ (Ch_Skill) เติมด้วยค่าเฉลี่ยเพื่อไม่ให้เสียจำนวน Sample
    df[feature_cols] = df[feature_cols].fillna(0)
    df['Ch_Skill'] = df['Ch_Skill'].fillna(df['Ch_Skill'].mean())
    
    # 5. สร้าง Target: 'Success_Target' (1 = ทักษะเพิ่มขึ้น, 0 = ทักษะเท่าเดิมหรือลดลง)
    df['Success_Target'] = (df['Ch_Skill'] > 0).astype(int)
    
    return df, feature_cols

# เรียกใช้ฟังก์ชัน
df_cleaned, features = clean_agricultural_data(df)

df_cleaned.head()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
# import shap # <--- สำหรับรันในเครื่องส่วนตัวที่มี library นี้

# 1. สร้างโมเดลจำแนกประเภท (Classification)
X = df_cleaned[features]
y = df_cleaned['Success_Target']
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# 2. การสร้างกราฟ Feature Importance (เพื่อดูการจัดลำดับปัจจัย)
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 8))
plt.title('Ranking of Factors for Farmer Skill Improvement')
plt.barh(range(len(indices)), importances[indices], color='teal', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance Score')
plt.tight_layout()
plt.show()

# 3. โค้ดสำหรับ SHAP (แนะนำให้ใช้ในเล่มวิจัย)
# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(X)
# shap.summary_plot(shap_values[1], X) # สรุปปัจจัยบวก/ลบต่อการพัฒนาทักษะ

In [ ]:
# --- LOGIC เดิมสำหรับการทำ Farmer Segmentation & Classification ---
# เป้าหมาย: พยากรณ์โอกาสความสำเร็จรายบุคคลเพื่อเลือกหลักสูตรที่เหมาะสม

import xgboost as xgb # หรือใช้ RandomForest แทนได้

# นิยามกลุ่มตัวแปรตามตรรกะเศรษฐศาสตร์
human_capital = ['age', 'edu', 'agri_long']
production_assets = ['land_crop1_in', 'irriga', 'gov_support']
financial_risk = ['loan', 'Non-Farm Act', 'Overview_Risk']
social_capital = ['agri Org_mem']
baseline_skills = ['Avg_ProdManage', 'Avg_InputManage', 'Avg_Tech', 'Avg_Ana&Plan', 'Avg_Mkting', 'Avg_Network']

all_features = human_capital + production_assets + financial_risk + social_capital + baseline_skills

# การสร้างโมเดลเพื่อใช้พยากรณ์ล่วงหน้า (Predictive Model)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
final_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.05)
final_model.fit(X_train, y_train)

# การวัดผล (Evaluation Metrics)
print(classification_report(y_test, final_model.predict(X_test)))